# Seeing in 3D with OpenCV and MiDaS

You have two eyes, and your brain compares what each one sees to work out how far away things are. A webcam has only one "eye", so it sees the world as a flat picture. Can a computer still tell what is near and what is far?

It can, with a **depth estimation** model. A depth model looks at a single picture and makes a **depth map**: a grayscale picture the same shape as the frame, where every pixel is bright if that part of the scene is close to the camera, and dark if it is far away.

The model learned this by studying millions of photos. It picks up on the same clues an artist uses to make a flat painting look 3D: things further away look smaller, closer objects cover up the ones behind them, and textures like grass or carpet get finer with distance.

In this notebook, we use **MiDaS**, a small depth model from Intel's research lab. We will download the model from Hugging Face, load it into OpenCV, create a depth function, try it on a photo, and then run it live on the webcam.

## Downloading the MiDaS model from Hugging Face

First, we need to download the model from Hugging Face. The following cell downloads a file called `model.onnx`. This file is a neural network model in ONNX format. ONNX stands for Open Neural Network eXchange and is one of the most popular formats for small models running in the browser or on hardware devices.

The file is about 33 MB, so the download may take a few moments.

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="simonguest/midas_v21_small_256x256-ONNX", filename="onnx/model.onnx")

## Loading the model

With the ONNX file downloaded, we now load the model into OpenCV using the `readNet` method in OpenCV's `dnn` module.

In [ ]:
import cv2

depth_net = cv2.dnn.readNet(model_path)
print("Neural network loaded")

## The estimate_depth function

The code in the next cell defines a new function called `estimate_depth`. You pass it a picture, and it returns a depth map.

The model doesn't measure distance in meters or feet. It only knows which parts of the picture are **nearer** or **further** than others. So the function scales its answer to run from `0` for the furthest thing in the picture to `1` for the nearest.

If you are interested in seeing how it works, expand the cell to view the code!

In [ ]:
import numpy as np

INPUT_SIZE = 256

def estimate_depth(frame):
  """Run MiDaS on a BGR frame. Returns a 256x256 depth map with values
  from 0 (the furthest thing in the picture) to 1 (the nearest)."""
  # 1. Squash the whole frame to 256x256 and turn it into a blob: RGB
  #    pixels from 0 to 1, in channel x height x width order.
  blob = cv2.dnn.blobFromImage(frame, 1 / 255, (INPUT_SIZE, INPUT_SIZE), swapRB=True)

  # 2. Run the network. It gives one number per pixel: bigger means closer.
  depth_net.setInput(blob)
  depth = depth_net.forward()[0]

  # 3. Rescale so the nearest pixel is 1 and the furthest is 0.
  return (depth - depth.min()) / (depth.max() - depth.min() + 1e-6)

## Trying it on a photo

Before we turn on the webcam, let's try the model on one of the pictures built into Codetto. The `colorize` function below turns the depth map into a heat map with OpenCV's `applyColorMap`: near things glow yellow and white, and far things fade to purple and black.

Run the cell to see the photo on the left and its depth map on the right. Then try changing `samples.Images.Dog` to `samples.Images.City` or `samples.Images.Forest`. Do the depth maps match what you would expect?

In [ ]:
from codetto import graphics2d, samples

def colorize(depth):
  """Turn a depth map (0 = far, 1 = near) into an RGB heat map."""
  gray = (depth * 255).astype(np.uint8)
  colored = cv2.applyColorMap(gray, cv2.COLORMAP_INFERNO)
  return cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)

photo = cv2.imread(samples.Images.Dog)
depth = estimate_depth(photo)

canvas = graphics2d.canvas(512, 256)
canvas.draw_pixels(cv2.cvtColor(cv2.resize(photo, (256, 256)), cv2.COLOR_BGR2RGB), 0, 0)
canvas.draw_pixels(colorize(depth), 256, 0)

## Start the webcam!

Now we'll start the webcam, capture frames in a loop, and pass each one to `estimate_depth`. The heat map is drawn over the whole canvas, so instead of the camera picture you see how far away everything is.

Try moving your hand towards the camera and away again, or leaning back in your chair. The heat map updates several times a second.

Press **Stop** when you are done.

In [ ]:
from codetto import graphics2d, webcam

canvas = graphics2d.canvas()
camera = webcam.start(canvas, mirror=True)
width, height = canvas.get_width(), canvas.get_height()
try:
  while True:
    frame = camera.capture_array()
    depth = estimate_depth(frame)
    canvas.draw_pixels(colorize(depth), 0, 0, width, height)
finally:
  camera.stop()

## Adding fog

The depth map doesn't have to replace the picture. `canvas.draw_mask` tints the camera picture, more strongly wherever the mask is bright. If we pass it `1 - depth`, the mask is bright for far things and dark for near things, so the tint lands on the background and leaves you clear.

With a pale gray color, this looks like fog rolling in behind you. Real fog works the same way: the further away something is, the more fog there is between you and it, so the paler it looks.

Run the cell, then press **Stop** and try a different `color` (`'#000000'` turns the fog into darkness, like a spotlight on you) or a different `opacity` (from `0.0` to `1.0`).

In [ ]:
from codetto import graphics2d, webcam

canvas = graphics2d.canvas()
camera = webcam.start(canvas, mirror=True)
try:
  while True:
    frame = camera.capture_array()
    depth = estimate_depth(frame)
    canvas.draw_mask(1 - depth, color='#e0e0e0', opacity=0.9)
finally:
  camera.stop()

## Replacing the background by distance

We can also make a mask that is simply *on* or *off*. `depth < NEAR` is `True` for every pixel further away than `NEAR` and `False` for the rest. Pass that mask to `canvas.draw_mask` with `image=`, and everything in the background is swapped for a picture.

Unlike a segmentation model, which only knows about people, this works for anything close to the camera. Hold up a book, a mug, or a pet, and it stays in the picture too.

Run the cell, then press **Stop** and try changing `NEAR`. A smaller number keeps more of the room, a bigger number keeps only what is closest to the camera.

In [ ]:
from codetto import graphics2d, samples, webcam

NEAR = 0.5

canvas = graphics2d.canvas()
camera = webcam.start(canvas, mirror=True)
try:
  while True:
    frame = camera.capture_array()
    depth = estimate_depth(frame)
    canvas.draw_mask(depth < NEAR, image=samples.Images.Undersea, opacity=0.95)
finally:
  camera.stop()

# Check Your Understanding